# ML Training Workflow

End-to-end LightGBM training using pipeline-generated features and triple-barrier labels.

1. **Setup** - Configure markets, date range, and label parameters
2. **Load Data** - Read features and labels from parquet store
3. **Explore Data** - Check distributions, missing values, correlations
4. **Preprocess** - Handle NaN, split train/test
5. **Train** - LightGBM with early stopping
6. **Evaluate** - Metrics, ROC, threshold analysis
7. **Feature Importance** - Identify top predictors

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from training import Dataset, LGBMTrainer, Evaluator
from qate.core.model import Market, ExchangeName, Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from triple_barrier.triple_barrier import TripleBarrierConfig

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

print('Setup complete!')

## Parameters

Configure markets, date range, and triple-barrier config.
The `TripleBarrierConfig` must match what was used in the labeling pipeline so that `param_id` resolves to the correct parquet partition.

In [ ]:
# Markets
FEATURE_MARKET = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
LABEL_MARKET = Market(ExchangeName.GMO, Symbol.XRP_JPY)
EXECUTION_SIZE = '150'

TIMEZONE = 'Asia/Tokyo'

START_DATE = "20251217"
END_DATE = "20251220"
DT_RANGE = DtRange.from_strings(START_DATE, END_DATE)

# Triple barrier config -- must match pipeline parameters
_BARRIER_MULTIPLIER = 1.5
_RETURN_SHIFT_SEC = 180
_TIMEOUT_SEC = _RETURN_SHIFT_SEC

CONFIG = TripleBarrierConfig(
    barrier_multiplier=_BARRIER_MULTIPLIER,
    return_shift_sec=_RETURN_SHIFT_SEC,
    timeout_sec=_TIMEOUT_SEC,
)

TARGET_COLUMN = 'label'

print(f'Feature market: {FEATURE_MARKET.id}')
print(f'Label market:   {LABEL_MARKET.id}')
print(f'Date range:     {DT_RANGE.start_str} -> {DT_RANGE.end_str}')
print(f'Label param_id: {CONFIG.param_id}')

## 1. Load Data

Read features and labels from parquet, normalize timezones, and convert
triple-barrier labels (`1`=Win, `-1`=Loss, `0`=Timeout) to binary classification.

In [ ]:
parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

# Load features
feature_store = parquet_store.create_feature(FEATURE_MARKET)
features_df = feature_store.read(DT_RANGE)

# Load labels (param_id must match the pipeline run)
label_store = parquet_store.create_label('TripleBarrierLabel', LABEL_MARKET, CONFIG.param_id)
labels_df = label_store.read(DT_RANGE)

print(f'Features: {features_df.shape[0]:,} rows x {features_df.shape[1]} columns')
print(f'Labels:   {labels_df.shape[0]:,} rows x {labels_df.shape[1]} columns')

In [ ]:
def prepare_df(df):
    """Ensure DatetimeIndex in UTC, handle ts column if present."""
    if 'ts' in df.columns:
        df.index = pd.to_datetime(df['ts'], unit='s')
        df = df.drop(columns=['ts'])
    # if df.index.tz is None:
    #     df.index = df.index.tz_localize('UTC')
    # elif str(df.index.tz) != 'UTC':
    #     df.index = df.index.tz_convert('UTC')
    return df.sort_index()

features_df = prepare_df(features_df)
labels_df = prepare_df(labels_df)

print(f'\nFeatures range: {features_df.index.min()} -> {features_df.index.max()}')
print(f'Labels range:   {labels_df.index.min()} -> {labels_df.index.max()}')

# Original label distribution
print('Triple-barrier label distribution:')
print(labels_df[TARGET_COLUMN].value_counts().sort_index().to_string())
print(f'  1 = Win, -1 = Loss, 0 = Timeout')

In [ ]:
# Convert to binary classification: Win(1)->1, Loss(-1)->0, drop Timeout(0)
labels_df = labels_df[labels_df[TARGET_COLUMN] != 0].copy()
labels_df[TARGET_COLUMN] = (labels_df[TARGET_COLUMN] == 1).astype(int)

print('Binary label distribution (after dropping timeouts):')
print(labels_df[TARGET_COLUMN].value_counts().sort_index().to_string())
print(f'\nWin rate: {labels_df[TARGET_COLUMN].mean():.2%}')
print(f'Samples:  {len(labels_df):,}')

In [ ]:
# Create Dataset and align features with labels
dataset = Dataset(
    features_df=features_df,
    labels_df=labels_df,
    target_column=TARGET_COLUMN,
)
dataset.align()

X, y = dataset.get_aligned_data()
print(f'Aligned dataset: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Label distribution after alignment:')
print(y.value_counts().sort_index().to_string())

## 2. Explore Data

In [ ]:
# Missing values
na_pct = (X.isna().sum() / len(X) * 100).sort_values(ascending=False)
na_nonzero = na_pct[na_pct > 0]

if len(na_nonzero) > 0:
    print(f'{len(na_nonzero)} columns with missing values (top 20):')
    print(na_nonzero.head(20).to_string())
else:
    print('No missing values.')

In [ ]:
# Feature statistics
X.describe().T

In [ ]:
# Target distribution over time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

y.value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('Class (0=Loss, 1=Win)')
axes[0].set_ylabel('Count')

y.rolling(500).mean().plot(ax=axes[1], alpha=0.8)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
axes[1].set_title('Win Rate Over Time (rolling 500)')
axes[1].set_ylabel('Win Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with target (top 20)
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print('Top 20 features correlated with target:')
print(correlations.head(20).to_string())

In [ ]:
# Correlation heatmap of top features
top_features = correlations.head(15).index.tolist()

plt.figure(figsize=(12, 10))
corr_matrix = X[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix (Top 15 by Target Correlation)')
plt.tight_layout()
plt.show()

## 3. Preprocess Data

In [ ]:
# Drop columns with >50% missing, forward-fill the rest
dataset.drop_na_features(threshold=0.5)
dataset.fill_na(method='ffill')

X, y = dataset.get_aligned_data()
print(f'After preprocessing: {X.shape[0]:,} rows, {X.shape[1]} features')
print(f'Remaining NaN: {X.isna().sum().sum()}')

In [ ]:
# Time-series train/test split (no shuffle)
dataset.train_test_split(test_size=0.2, shuffle=False)

X_train, y_train = dataset.get_train_data()
X_test, y_test = dataset.get_test_data()

print(f'Train: {len(X_train):,} samples  ({X_train.index.min()} -> {X_train.index.max()})')
print(f'Test:  {len(X_test):,} samples  ({X_test.index.min()} -> {X_test.index.max()})')
print(f'\nTrain win rate: {y_train.mean():.2%}')
print(f'Test  win rate: {y_test.mean():.2%}')

## 4. Train Model

In [ ]:
trainer = LGBMTrainer(
    task='classification',
    params={
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42,
    },
    num_boost_round=1000,
    early_stopping_rounds=50,
    verbose_eval=100,
)

trainer.fit(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
)

print(f'\nBest iteration: {trainer.best_iteration}')

In [ ]:
# Training history
if trainer.training_history:
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, metrics in trainer.training_history.items():
        for metric, values in metrics.items():
            ax.plot(values, label=f'{name}_{metric}')
    if trainer.best_iteration:
        ax.axvline(x=trainer.best_iteration, color='r', linestyle='--',
                   label=f'Best iter ({trainer.best_iteration})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Training History')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Evaluate Model

In [ ]:
evaluator = Evaluator(trainer, threshold=0.5)
results = evaluator.evaluate(X_test, y_test, return_predictions=True)
print(results.summary())

In [ ]:
# Confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(
    results.confusion_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Loss', 'Pred Win'],
    yticklabels=['Actual Loss', 'Actual Win'],
)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if results.roc_curve:
    auc_val = results.metrics.get('roc_auc', 0)
    axes[0].plot(results.roc_curve['fpr'], results.roc_curve['tpr'],
                 'b-', label=f'ROC (AUC={auc_val:.4f})')
    axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve')
    axes[0].legend()

if results.pr_curve:
    axes[1].plot(results.pr_curve['recall'], results.pr_curve['precision'], 'b-')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

In [ ]:
# Optimal threshold search
threshold_results = evaluator.find_best_threshold(X_test, y_test, metric='f1')

print(f"Best threshold: {threshold_results['best_threshold']:.2f}")
print('Metrics at best threshold:')
for k, v in threshold_results['best_metrics'].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Threshold analysis plot
threshold_df = threshold_results['all_results']

plt.figure(figsize=(10, 6))
for col in ['accuracy', 'precision', 'recall', 'f1']:
    plt.plot(threshold_df['threshold'], threshold_df[col], label=col)
plt.axvline(x=threshold_results['best_threshold'], color='r', linestyle='--',
            label=f"Best F1 ({threshold_results['best_threshold']:.2f})")
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Classification Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importance_df = trainer.feature_importance(importance_type='gain')
print('Top 20 important features:')
importance_df.head(20)

In [ ]:
top_n = min(20, len(importance_df))
top_imp = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_imp['importance'].values)
plt.yticks(range(top_n), top_imp['feature'].values)
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Prediction Analysis

In [ ]:
analysis = evaluator.analyze_predictions(X_test, y_test)

print('Prediction categories:')
print(analysis['category'].value_counts().to_string())

print('\nAccuracy by confidence level:')
analysis['confidence_bin'] = pd.cut(analysis['confidence'], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
print(analysis.groupby('confidence_bin', observed=True)['correct'].agg(['mean', 'count']).to_string())

In [ ]:
# Probability distribution by actual class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in [0, 1]:
    mask = analysis['y_true'] == label
    axes[0].hist(analysis.loc[mask, 'probability'], bins=50, alpha=0.5,
                 label=f'Class {label} ({"Loss" if label == 0 else "Win"})')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability Distribution by Actual Class')
axes[0].legend()

# Calibration plot
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(analysis['y_true'], analysis['probability'], n_bins=10)
axes[1].plot(prob_pred, prob_true, 's-', label='Model')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predictions over time
analysis_ts = analysis.copy()
analysis_ts.index = X_test.index

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

window = 100
axes[0].plot(analysis_ts.index, analysis_ts['y_true'].rolling(window).mean(),
             label='Actual', alpha=0.7)
axes[0].plot(analysis_ts.index, analysis_ts['probability'].rolling(window).mean(),
             label='Predicted', alpha=0.7)
axes[0].set_ylabel('Value')
axes[0].set_title(f'Actual vs Predicted (Rolling {window})')
axes[0].legend()

rolling_acc = analysis_ts['correct'].rolling(window).mean()
axes[1].plot(analysis_ts.index, rolling_acc)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'Rolling Accuracy (window={window})')

axes[2].plot(analysis_ts.index, analysis_ts['confidence'].rolling(window).mean())
axes[2].set_ylabel('Confidence')
axes[2].set_title(f'Rolling Confidence (window={window})')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

## 8. Cross-Validation (Optional)

In [ ]:
X_all, y_all = dataset.get_aligned_data()
cv_results = evaluator.cross_validate(X_all, y_all, n_splits=5)

cv_summary = pd.DataFrame(cv_results)
print('Cross-Validation Summary:')
print(cv_summary.describe().T[['mean', 'std']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_summary.plot(kind='box', ax=ax)
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Results')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
MODEL_PATH = '../models/lgbm_model'
trainer.save(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# Verify load
loaded_trainer = LGBMTrainer.load(MODEL_PATH)
original_preds = trainer.predict(X_test)
loaded_preds = loaded_trainer.predict(X_test)

match = np.allclose(original_preds, loaded_preds)
print(f'Predictions match after reload: {match}')